# Lezione 2 · La piccola rete

Due neuroni nascosti con la ReLU, un'uscita con la sigmoide, e la softmax con la temperatura.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

## La rete 3 → 2 → 1

Il primo neurone nascosto, h₁, è lo «slancio»; il secondo, h₂, è la «pioggia che pesa».

In [2]:
W1 = np.array([[ 2.0, 0.0, 1.0],    # h1: energia, pioggia, compagno
               [-3.0, 2.0, 0.0]])   # h2
b1 = np.array([-1.0, 0.5])
W2 = np.array([1.5, -2.0])
b2 = -0.5

def relu(v):
    return np.maximum(0, v)

def rete(x):
    h = relu(W1 @ x + b1)
    z = W2 @ h + b2
    return h, z, sigmoide(z)

h, z, p = rete(np.array([0.8, 1, 1]))
controlla("h1", h[0], 1.6)
controlla("h2", h[1], 0.1)
controlla("z", z, 1.7)
controlla("probabilità", p, 0.85)

✓  h1: calcolato 1.6, nel libro 1.6
✓  h2: calcolato 0.1, nel libro 0.1
✓  z: calcolato 1.7, nel libro 1.7
✓  probabilità: calcolato 0.8455, nel libro 0.85


## Kata per casa

**Esercizio 1.** La softmax dei logit (3; 1; 0), prima normale e poi con temperatura 2.

In [3]:
def softmax(logit, T=1.0):
    e = np.exp(np.array(logit) / T)
    return e / e.sum()

p1 = softmax([3, 1, 0])
for nome, val, atteso in zip(["opzione 1", "opzione 2", "opzione 3"], p1, [0.84, 0.11, 0.04]):
    controlla(nome, val, atteso)

p2 = softmax([3, 1, 0], T=2)
for nome, val, atteso in zip(["opzione 1, T=2", "opzione 2, T=2", "opzione 3, T=2"], p2, [0.63, 0.23, 0.14]):
    controlla(nome, val, atteso)

✓  opzione 1: calcolato 0.8438, nel libro 0.84
✓  opzione 2: calcolato 0.1142, nel libro 0.11
✓  opzione 3: calcolato 0.04201, nel libro 0.04
✓  opzione 1, T=2: calcolato 0.6285, nel libro 0.63
✓  opzione 2, T=2: calcolato 0.2312, nel libro 0.23
✓  opzione 3, T=2: calcolato 0.1402, nel libro 0.14


**Esercizio 2.** La serata (0,3; 1; 0): stanco e sotto la pioggia.

In [4]:
h, z, p = rete(np.array([0.3, 1, 0]))
controlla("h1 (spento)", h[0], 0.0)
controlla("h2", h[1], 1.6)
controlla("z", z, -3.7)
controlla("probabilità", p, 0.02)

✓  h1 (spento): calcolato 0, nel libro 0
✓  h2: calcolato 1.6, nel libro 1.6
✓  z: calcolato -3.7, nel libro -3.7
✓  probabilità: calcolato 0.02413, nel libro 0.02


## Prova tu

Alza e abbassa la temperatura: che cosa succede alla scelta più probabile quando T va verso zero? E quando cresce molto?

In [5]:
for T in [0.25, 0.5, 1, 2, 5]:
    print(f"T = {T}: " + ", ".join(f"{v:.0%}" for v in softmax([3, 1, 0], T)))

T = 0.25: 100%, 0%, 0%
T = 0.5: 98%, 2%, 0%
T = 1: 84%, 11%, 4%
T = 2: 63%, 23%, 14%
T = 5: 45%, 30%, 25%
